In [0]:
from utils.api_client import fetch_crypto_data
from utils.spark_utils import get_spark
from utils.logger import get_logger
from pyspark.sql.functions import current_timestamp

In [0]:
spark = get_spark()
logger = get_logger()

In [0]:
import json

with open("../../configs/entities.json") as f:
    config = json.load(f)

config = config['entities'][0]

In [0]:
logger.info("Starting Bronze Layer")
data = fetch_crypto_data(config["coins"])
data

In [0]:
import pandas as pd

df = pd.DataFrame(data)
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
df.head()

In [0]:
spark_df = spark.createDataFrame(df)
spark_df = spark_df.withColumn("ingestion_time", current_timestamp())
spark_df.show(5)

In [0]:
# spark_df.write.format("delta") \
#     .mode("append") \
#     .save("/home/jovyan/CryptoInsight")

In [0]:
# Improve Your Batch Design 
# Add: Partition by date
# 🔥 Why this matters:
# - Faster queries
# - Efficient incremental loads
# - Scalable storage

from pyspark.sql.functions import to_date

spark_df = spark_df.withColumn("date", to_date("timestamp"))

spark_df.show(5)

In [0]:
# spark_df.write.partitionBy("coin") \
#     .format("delta") \
#     .mode("append") \
#     .save("dbfs:/Volumes/workspace/cryptoinsight/bronze")

# spark_df.write.option("header", "true").format("csv").save("bronze")

In [0]:
spark_df.write.partitionBy("coin") \
    .format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable(config["bronze_table"])

In [0]:
%sql
OPTIMIZE workspace.cryptoinsight.bronze
ZORDER BY (date)

In [0]:
logger.info("Bronze Load Complete")